# Parte 2: Preprocesamiento de datos

En este código realizaremos la limpieza de los registros descargados directamente desde la página OMDB.

In [ ]:
#Importación de las librerías necesarias
import warnings
warnings.filterwarnings('ignore')
from tqdm import tqdm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPool2D
from tensorflow.keras.preprocessing import image
import requests
from PIL import Image
import io
import pickle

Aquí usaremos el archivo de movies, el cual contiene todos los registros permitidos por la descarga, se extraerán las columnas que nos interesan, que es el id, el género y el poster.

In [ ]:
#Leemos el dataset
with open('/content/drive/MyDrive/PIA/movies.csv', encoding='utf-8') as f:
    movies = pd.read_csv(f, sep=',')

In [ ]:
#Extraemos las columnas necesarias
columnas = ['imdbID', 'Genre', 'Poster']
movie_cf = movies[columnas]
movie_cf

,imdbID,Genre,Poster
0,tt1239216,"Drama, Family",https://m.media-amazon.com/images/M/MV5BZGJkZj...
1,tt0124272,"Adventure, Family",https://m.media-amazon.com/images/M/MV5BMGJlOW...
2,tt1210840,Comedy,https://m.media-amazon.com/images/M/MV5BMTI4Nj...
3,tt9650040,"Biography, Music",https://m.media-amazon.com/images/M/MV5BODk4Ym...
4,tt1075747,"Action, Drama, Fantasy",https://m.media-amazon.com/images/M/MV5BMTQ2Nz...
...,...,...,...
127413,tt6325170,Drama,https://m.media-amazon.com/images/M/MV5BMTkxOT...
127414,tt6324988,Reality-TV,https://m.media-amazon.com/images/M/MV5BODA1YT...
127415,tt6324892,Comedy,https://m.media-amazon.com/images/M/MV5BYTc4MW...
127416,tt6324856,Crime,https://m.media-amazon.com/images/M/MV5BNTFhOD...


Observando el csv, hubo géneros que no consideré necesarios para el estudio o que incluían muy pocos registros a comparación de otros como comedia o drama, por lo que se eliminan.

In [ ]:
#Eliminamos géneros que no quiero que salgan
drop_generos = ['Biography', 'Game-show', 'Game-Show', 'News', 'Reality-TV', 'Talk-Show', 'Western', 'Documentary', 'Short', 'History', 'Musical', 'War']
movie_cf = movie_cf[~movie_cf['Genre'].str.contains('|'.join(drop_generos))]
movie_cf = movie_cf.reset_index(drop=True)

Se separan los géneros por columnas donde cada columna es una bandera de 1 y 0, donde el 1 significa que pertenece a esa categoría y 0 que no pertenece a esa categoría. También se remueven duplicados.

In [ ]:
#Obtenemos los diferentes géneros de las películas y las separamos en columnas
movie_cf['Genre'] = movie_cf['Genre'].str.replace(' ', '')
movie_cf = movie_cf.drop_duplicates()
generos_separados = movie_cf['Genre'].str.get_dummies(sep=',')
generos_separados.sum()

Action        7227
Adventure     3940
Animation     1649
Comedy       14441
Crime         5800
Drama        22459
Family        2767
Fantasy       2320
Horror        6594
Music         2206
Mystery       3084
Romance       5043
Sci-Fi        2915
Sport         1476
Thriller      7575
dtype: int64

In [ ]:
movie_cf.shape[0]

48139

In [ ]:
#Concatenamos las columnas al dataframe
movie_cf = pd.concat([movie_cf, generos_separados], axis=1)
movie_cf.sample(5)

,imdbID,Genre,Poster,Action,Adventure,Animation,Comedy,Crime,Drama,Family,Fantasy,Horror,Music,Mystery,Romance,Sci-Fi,Sport,Thriller
10567,tt0080158,"Drama,Music",https://m.media-amazon.com/images/M/MV5BZDI1MD...,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0
32543,tt0388377,"Action,Adventure,Drama",https://m.media-amazon.com/images/M/MV5BM2VhMG...,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0
41636,tt8718114,Drama,https://m.media-amazon.com/images/M/MV5BNTI4Mz...,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
19015,tt0109318,Drama,https://m.media-amazon.com/images/M/MV5BMjU2ND...,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
14832,tt0095350,Horror,https://m.media-amazon.com/images/M/MV5BZWYxNm...,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0


Se descarga la imagen contenida en el link de la columna poster de cada registro, si existe; se ajusta el tamaño a 250x250 y se guarda localmente. Este paso también se hizo en etapas para cuidar los recursos de la computadora. 

Como aclaración, debo indicar que este código fue ejecutado desde google colab, puesto que era más rápida la descarga de los links que localmente en mi computadora. Es por eso que la ruta donde va a guardar las imágenes es diferende de la ruta donde las lee en las siguientes partes.

In [ ]:
#Definimos el array donde se van a ir guardando las imágenes
imagenes = []
movie_leido = pd.DataFrame()
columnas_movie_cf = movie_cf.columns
movie_leido = pd.DataFrame(columns=columnas_movie_cf)
indices = []

In [ ]:
#Tratando las imágenes
index = 0
for i in tqdm(range(0, 7179)):
    try:
        path = movie_cf['Poster'][i]
        name = movie_cf['imdbID'][i]
        response = requests.get(path)
        imagen = Image.open(io.BytesIO(response.content))
        imagen_resized = imagen.resize((250, 250))  # Redimensionar la imagen si es necesario
        imagen_resized.save(f'/content/drive/MyDrive/PIA3/{name}.jpg')
        index+=1
    except Exception as e:
        indices.append(i)
        continue

100%|██████████| 7179/7179 [12:35<00:00,  9.50it/s]


In [ ]:
#Tratando las imágenes
for i in tqdm(range(7179, 15000)):
  try:
    path = movie_cf['Poster'][i]
    name = movie_cf['imdbID'][i]
    response = requests.get(path)
    imagen = Image.open(io.BytesIO(response.content))
    imagen_resized = imagen.resize((250, 250))  # Redimensionar la imagen si es necesario
    imagen_resized.save(f'/content/drive/MyDrive/PIA3/{name}.jpg')
    index+=1
  except Exception as e:
    indices.append(i)
    continue

100%|██████████| 7821/7821 [23:52<00:00,  5.46it/s]


In [ ]:
#Tratando las imágenes
for i in tqdm(range(15000, movie_cf.shape[0])):
  try:
      path = movie_cf['Poster'][i]
      name = movie_cf['imdbID'][i]
      response = requests.get(path)
      imagen = Image.open(io.BytesIO(response.content))
      imagen_resized = imagen.resize((250, 250))  # Redimensionar la imagen si es necesario
      imagen_resized.save(f'/content/drive/MyDrive/PIA3/{name}.jpg')
      index+=1
  except Exception as e:
      indices.append(i)
      continue

100%|██████████| 33139/33139 [1:45:33<00:00,  5.23it/s]


Por último ya con todas las imágenes descargadas, se guarda un nuevo csv con las películas que oficialmente se van a analizar.

In [ ]:
movie_cf = movie_cf.drop(indices)
movie_cf.to_csv('Películas con imagen.csv', index=False)